# GPR92 / LPAR5 mapping in pancreas — cleaned analysis notebook

This notebook is part of an integrated workflow (neonatal → chronic pancreatitis → spatial → integrated spatial).

**Local data (not committed):**
- `../data/raw/`
- `../data/processed/`

Outputs are written to `../outputs/`.

In [ ]:
# --- Environment & paths (portable) ---
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
OUT_DIR = Path("../outputs")
(OUT_DIR / "figures").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "tables").mkdir(parents=True, exist_ok=True)

# Local project modules
import sys
sys.path.append(str(Path("..").resolve()))
from src import utils, preprocessing, scoring, plotting, spatial

In [ ]:
from pathlib import Path
import os
import gzip
DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
OUT_DIR = Path("../outputs")

print("Exists:", GSE_DIR.exists())
print("Top-level files/folders (first 50):")
items = sorted(GSE_DIR.iterdir(), key=lambda x: x.name)
for p in items[:50]:
    kind = "DIR " if p.is_dir() else "FILE"
    size_mb = (p.stat().st_size / 1e6) if p.is_file() else None
    print(f"  {kind:4}  {p.name}" + (f"  ({size_mb:.2f} MB)" if size_mb else ""))

# Find common 10x-style files recursively
patterns = ["matrix.mtx", "matrix.mtx.gz", "features.tsv", "features.tsv.gz",
            "genes.tsv", "genes.tsv.gz", "barcodes.tsv", "barcodes.tsv.gz"]

hits = []
for pat in patterns:
    hits.extend(list(GSE_DIR.rglob(pat)))

print("\n10x-like file hits:")
for h in hits[:50]:
    print(" ", h.relative_to(GSE_DIR))
print(f"Total 10x-like hits: {len(hits)}")

# Find any big text-like expression/metadata files
big_files = []
for p in GSE_DIR.rglob("*"):
    if p.is_file() and p.suffix in [".txt", ".tsv", ".csv", ".gz", ".mtx", ".h5", ".h5ad"]:
        if p.stat().st_size > 50e6:  # >50MB
            big_files.append(p)

big_files = sorted(big_files, key=lambda x: x.stat().st_size, reverse=True)
print("\nLargest candidate data files (top 20):")
for p in big_files[:20]:
    print(f"  {p.relative_to(GSE_DIR)}  ({p.stat().st_size/1e6:.1f} MB)")

# If there is a gz text file, peek first 8 lines (safe)
def peek_gz_text(gz_path, n=8):
    print(f"\n--- Peek: {gz_path.name} ---")
    with gzip.open(gz_path, "rt", errors="ignore") as f:
        for _ in range(n):
            line = f.readline()
            if not line:
                break
            print(line.rstrip())

# Peek a few small .gz files that look like metadata (not matrices)
gz_candidates = [p for p in GSE_DIR.rglob("*.gz") if p.stat().st_size < 5e6]  # <5MB
for p in gz_candidates[:3]:
    # only peek if it seems text-like
    if any(x in p.name.lower() for x in ["meta", "annot", "barcode", "feature", "gene", "cell", "cluster"]):
        peek_gz_text(p, 8)


In [ ]:
from pathlib import Path
import pandas as pd
DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
OUT_DIR = Path("../outputs")

matrix_files = sorted(GSE_DIR.glob("*_final_matrix.csv.gz"))
print("Found matrices:", [f.name for f in matrix_files])

def inspect_matrix(path, nrows=5, ncols=10):
    print("\n==============================")
    print("File:", path.name)

    # Read small preview
    df = pd.read_csv(path, compression="gzip", nrows=nrows)
    print("Preview shape (nrows):", df.shape)
    print("Columns (first 20):", list(df.columns[:20]))

    # Show first few rows (subset)
    display(df.iloc[:nrows, :min(ncols, df.shape[1])])

for f in matrix_files:
    inspect_matrix(f, nrows=5, ncols=12)


In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
import anndata as ad
from pathlib import Path
from scipy import sparse


In [ ]:
import pandas as pd
import numpy as np
import anndata as ad
from pathlib import Path
from scipy import sparse

def load_olaniru_long_matrix(path_gz: str | Path,
                            keep_feature_type: str = "Gene Expression",
                            use_gene_col: str = "feature_name") -> ad.AnnData:
    """
    Loads Olaniru 'final_matrix.csv.gz' (long table):
    cell_barcode, feature_id, feature_name, feature_type, count
    Builds sparse AnnData (cells x genes) for the selected feature_type.
    """
    path_gz = Path(path_gz)

    cols = ["cell_barcode", "feature_id", "feature_name", "feature_type", "count"]

    # Read everything as string first (robust), then convert count safely
    df = pd.read_csv(
        path_gz,
        compression="gzip",
        sep="\t",              # these look tab-delimited
        header=None,
        names=cols,
        dtype="string",
        engine="python"
    )

    # Drop obvious header-like rows (e.g., count == "count" or feature_type == "feature_type")
    df = df[df["count"].str.lower().ne("count")]
    df = df[df["feature_type"].str.lower().ne("feature_type")]

    # Convert count to numeric; coerce bad rows to NaN, then drop them
    df["count"] = pd.to_numeric(df["count"], errors="coerce")
    df = df.dropna(subset=["count"])
    df["count"] = df["count"].astype(np.int32)

    # Filter to Gene Expression (or Antibody Capture if desired)
    df = df[df["feature_type"] == keep_feature_type].copy()

    # Use gene symbols unless you want Ensembl IDs
    if use_gene_col == "feature_name":
        genes = df["feature_name"].astype("category")
    elif use_gene_col == "feature_id":
        genes = df["feature_id"].astype("category")
    else:
        raise ValueError("use_gene_col must be 'feature_name' or 'feature_id'")

    cells = df["cell_barcode"].astype("category")

    cell_codes = cells.cat.codes.to_numpy()
    gene_codes = genes.cat.codes.to_numpy()
    counts = df["count"].to_numpy()

    X = sparse.coo_matrix(
        (counts, (cell_codes, gene_codes)),
        shape=(cells.cat.categories.size, genes.cat.categories.size)
    ).tocsr()

    adata = ad.AnnData(X=X)
    adata.obs_names = cells.cat.categories.astype(str)
    adata.var_names = genes.cat.categories.astype(str)
    adata.var_names_make_unique()

    adata.obs["source_file"] = path_gz.name
    adata.obs["dataset"] = "Olaniru_GSE197064"
    return adata


In [ ]:
from pathlib import Path
import anndata as ad
DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
OUT_DIR = Path("../outputs")
matrix_files = sorted(GSE_DIR.glob("*_final_matrix.csv.gz"))

adatas = []
keys = []
for f in matrix_files:
    print("Loading:", f.name)
    a = load_olaniru_long_matrix(f, keep_feature_type="Gene Expression", use_gene_col="feature_name")
    print("  shape:", a.shape)
    adatas.append(a)
    keys.append(f.stem.replace("_final_matrix", ""))

adata_olaniru = ad.concat(adatas, label="batch", keys=keys, join="outer", fill_value=0)
adata_olaniru.var_names_make_unique()
print("Combined Olaniru shape:", adata_olaniru.shape)


In [ ]:
import scanpy as sc
sc.pp.calculate_qc_metrics(adata_olaniru, inplace=True)
print("LPAR5 present?", "LPAR5" in adata_olaniru.var_names)


In [ ]:
import gzip
from pathlib import Path

p = Path(r"../data/raw/GSM5907965_121313_final_matrix.csv.gz")

with gzip.open(p, "rt", errors="ignore") as f:
    for i in range(3):
        line = f.readline().rstrip("\n")
        print(f"LINE {i+1}:", line)
        print("  tab-split cols:", len(line.split("\t")))
        print("  comma-split cols:", len(line.split(",")))


In [ ]:
import pandas as pd
import numpy as np
import anndata as ad
from pathlib import Path
from scipy import sparse

def load_olaniru_long_matrix_csv(path_gz: str | Path,
                                keep_feature_type: str = "Gene Expression",
                                use_gene_col: str = "feature_name") -> ad.AnnData:
    """
    Loads Olaniru final_matrix.csv.gz (comma-separated long table):
    cell_barcode, feature_id, feature_name, feature_type, count
    -> returns AnnData (cells x genes) for Gene Expression rows.
    """
    path_gz = Path(path_gz)
    cols = ["cell_barcode", "feature_id", "feature_name", "feature_type", "count"]

    df = pd.read_csv(
        path_gz,
        compression="gzip",
        sep=",",
        header=None,
        names=cols,
        dtype={"cell_barcode": "string",
               "feature_id": "string",
               "feature_name": "string",
               "feature_type": "string",
               "count": "int32"}
    )

    # Clean whitespace (important)
    df["feature_type"] = df["feature_type"].str.strip()
    df["cell_barcode"] = df["cell_barcode"].str.strip()
    df["feature_name"] = df["feature_name"].str.strip()
    df["feature_id"] = df["feature_id"].str.strip()

    # Filter to Gene Expression only
    df = df[df["feature_type"] == keep_feature_type].copy()

    # Build sparse matrix
    cells = df["cell_barcode"].astype("category")
    genes = df[use_gene_col].astype("category")

    X = sparse.coo_matrix(
        (df["count"].to_numpy(),
         (cells.cat.codes.to_numpy(), genes.cat.codes.to_numpy())),
        shape=(cells.cat.categories.size, genes.cat.categories.size)
    ).tocsr()

    adata = ad.AnnData(X=X)
    adata.obs_names = cells.cat.categories.astype(str)
    adata.var_names = genes.cat.categories.astype(str)
    adata.var_names_make_unique()
    adata.obs["source_file"] = path_gz.name
    adata.obs["dataset"] = "Olaniru_GSE197064"
    return adata
DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
OUT_DIR = Path("../outputs")
matrix_files = sorted(GSE_DIR.glob("*_final_matrix.csv.gz"))

adatas = []
keys = []
for f in matrix_files:
    print("Loading:", f.name)
    a = load_olaniru_long_matrix_csv(f, keep_feature_type="Gene Expression", use_gene_col="feature_name")
    print("  shape:", a.shape)
    adatas.append(a)
    keys.append(f.stem.replace("_final_matrix", ""))

adata_olaniru = ad.concat(adatas, label="batch", keys=keys, join="outer", fill_value=0)
adata_olaniru.var_names_make_unique()

print("Combined Olaniru shape:", adata_olaniru.shape)
print("Batches:", adata_olaniru.obs["batch"].unique().tolist()[:10])


In [ ]:
import scanpy as sc

sc.pp.calculate_qc_metrics(adata_olaniru, inplace=True)

print("LPAR5 present?", "LPAR5" in adata_olaniru.var_names)
# (GPR92 alias should map to LPAR5; if not present, we’ll handle mapping later.)


In [ ]:
from pathlib import Path
import gzip

OMIX10X_DIR = Path(r"../data/raw/OMIX236-20-02")
OMIXMSTRT_DIR = Path(r"../data/raw/OMIX236-20-01")

def list_dir(p: Path, n=30):
    print("\n===", p, "===")
    print("Exists:", p.exists())
    if not p.exists():
        return
    items = sorted(p.rglob("*"))
    files = [x for x in items if x.is_file()]
    print("Total files:", len(files))
    for f in files[:n]:
        print(" ", f.relative_to(p), f"({f.stat().st_size/1e6:.2f} MB)")

list_dir(OMIXMSTRT_DIR)
list_dir(OMIX10X_DIR)

# Peek first 3 lines of the largest .txt or .gz in each folder
def peek_largest_text(folder: Path):
    candidates = [p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in [".txt", ".tsv", ".csv", ".gz"]]
    if not candidates:
        print("\nNo text/gz candidates in", folder)
        return
    candidates = sorted(candidates, key=lambda x: x.stat().st_size, reverse=True)
    p = candidates[0]
    print(f"\n--- Largest file in {folder.name}: {p.name} ({p.stat().st_size/1e6:.1f} MB) ---")

    if p.suffix.lower() == ".gz":
        with gzip.open(p, "rt", errors="ignore") as f:
            for i in range(3):
                line = f.readline().rstrip("\n")
                print(f"LINE {i+1}:", line)
                print("  tab cols:", len(line.split('\t')), "| comma cols:", len(line.split(',')))
    else:
        with open(p, "rt", errors="ignore") as f:
            for i in range(3):
                line = f.readline().rstrip("\n")
                print(f"LINE {i+1}:", line)
                print("  tab cols:", len(line.split('\t')), "| comma cols:", len(line.split(',')))

peek_largest_text(OMIXMSTRT_DIR)
peek_largest_text(OMIX10X_DIR)


In [ ]:
import numpy as np
import pandas as pd
import anndata as ad
from pathlib import Path
from scipy import sparse
import re

def parse_cellname_meta(cellnames):
    """Extract week and enrichment tags from OMIX cell names like W9_1_EpCAM_Pos_001."""
    week = []
    enrich = []
    sample = []
    for c in cellnames:
        m = re.match(r"^(W\d+)_([^_]+)_(.*)_(\d+)$", c)
        if m:
            week.append(m.group(1))          # W9, W18, etc
            sample.append(m.group(2))        # 1, 2, etc
            enrich.append(m.group(3))        # EpCAM_Pos, No_Enrichment, etc
        else:
            week.append(None)
            sample.append(None)
            enrich.append(None)
    obs = pd.DataFrame({"week": week, "sample": sample, "enrichment": enrich}, index=cellnames)
    return obs

def load_omix_umi_txt_sparse(path_txt: str | Path, chunk_genes: int = 200) -> ad.AnnData:
    """
    Load OMIX UMI.txt (tab-delimited wide matrix):
    ID, Symbol, cell1, cell2, ... cellN
    Returns AnnData with X (cells x genes) sparse CSR.
    """
    path_txt = Path(path_txt)

    # Read header only to get cell names
    header = pd.read_csv(path_txt, sep="\t", nrows=0)
    cols = header.columns.tolist()
    assert cols[0] == "ID" and cols[1] == "Symbol", "Unexpected first columns; expected ID and Symbol."
    cell_names = cols[2:]

    X_blocks = []
    var_ensembl = []
    var_symbol = []

    # Read in chunks (rows = genes)
    reader = pd.read_csv(
        path_txt,
        sep="\t",
        chunksize=chunk_genes,
        low_memory=False
    )

    for i, chunk in enumerate(reader, start=1):
        # chunk columns: ID, Symbol, many cell columns
        ensembl = chunk.iloc[:, 0].astype(str).values
        symbol  = chunk.iloc[:, 1].astype(str).values

        # numeric matrix: genes x cells
        vals = chunk.iloc[:, 2:].to_numpy(dtype=np.float32, copy=False)

        # Convert to sparse; then transpose to cells x genes
        block = sparse.csr_matrix(vals).T  # now (cells x genes_chunk)

        X_blocks.append(block)
        var_ensembl.extend(ensembl.tolist())
        var_symbol.extend(symbol.tolist())

        if i % 20 == 0:
            print(f"  loaded ~{i*chunk_genes} genes...")

    X = sparse.hstack(X_blocks, format="csr")  # (cells x all_genes)

    adata = ad.AnnData(X=X)
    adata.obs_names = pd.Index(cell_names, name="cell")
    adata.var_names = pd.Index(var_symbol, name="gene_symbol")
    adata.var["ensembl_id"] = var_ensembl
    adata.var_names_make_unique()

    # Add parsed metadata from cell names
    adata.obs = parse_cellname_meta(cell_names)

    return adata


In [ ]:
OMIX_MSTRT = Path(r"../data/raw/UMI.txt")
OMIX_10X   = Path(r"../data/raw/UMI.txt")

print("Loading Yu mSTRTSeq...")
adata_yu_mstrt = load_omix_umi_txt_sparse(OMIX_MSTRT, chunk_genes=200)
adata_yu_mstrt.obs["dataset"] = "Yu_mSTRTSeq"

print("Loading Yu 10x...")
adata_yu_10x = load_omix_umi_txt_sparse(OMIX_10X, chunk_genes=200)
adata_yu_10x.obs["dataset"] = "Yu_10x"

print("Yu mSTRT shape:", adata_yu_mstrt.shape)
print("Yu 10x shape:", adata_yu_10x.shape)


In [ ]:
from pathlib import Path

OMIX_MSTRT = Path(r"../data/raw/UMI.txt")
OMIX_10X   = Path(r"../data/raw/UMI.txt")

def show_header(path, n=12):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        header_line = f.readline().rstrip("\n")
    cols = [c.strip().lstrip("\ufeff") for c in header_line.split("\t")]
    print("\nFILE:", path.name)
    print("First columns:", cols[:n])

show_header(OMIX_MSTRT)
show_header(OMIX_10X)


In [ ]:
import numpy as np
import pandas as pd
import anndata as ad
from pathlib import Path
from scipy import sparse

def parse_cellname_meta_general(cellnames):
    """
    Handles both:
      - mSTRT: W9_1_EpCAM_Pos_001
      - 10x:   W8_1_AAACCTGAGATCTGCT
    Extracts week, sample, and rest (enrichment or barcode).
    """
    week = []
    sample = []
    tag = []
    for c in cellnames:
        parts = c.split("_", 2)  # split into at most 3 parts
        if len(parts) >= 2:
            week.append(parts[0])     # W8, W9, W18...
            sample.append(parts[1])   # 1, 2...
            tag.append(parts[2] if len(parts) == 3 else None)
        else:
            week.append(None); sample.append(None); tag.append(None)
    return pd.DataFrame({"week": week, "sample": sample, "tag": tag}, index=cellnames)

def load_omix_umi_txt_sparse_auto(path_txt: str | Path, chunk_genes: int = 200) -> ad.AnnData:
    """
    Loads OMIX UMI.txt (tab-delimited wide matrix) in either format:
      A) ID, Symbol, cell1, cell2, ...
      B) Symbol, cell1, cell2, ...
    Returns AnnData with X (cells x genes) sparse CSR.
    """
    path_txt = Path(path_txt)

    # Read header line safely (handles BOM)
    with open(path_txt, "r", encoding="utf-8", errors="ignore") as f:
        header_line = f.readline().rstrip("\n")
    cols = [c.strip().lstrip("\ufeff") for c in header_line.split("\t")]

    # Detect format
    fmt = None
    if len(cols) >= 3 and cols[0] == "ID" and cols[1] == "Symbol":
        fmt = "ID_SYMBOL"
        gene_id_idx = 0
        gene_symbol_idx = 1
        cell_start_idx = 2
    elif len(cols) >= 2 and cols[0] == "Symbol":
        fmt = "SYMBOL_ONLY"
        gene_id_idx = None
        gene_symbol_idx = 0
        cell_start_idx = 1
    else:
        # fallback: assume first col is Symbol, rest are cells
        fmt = "FALLBACK_SYMBOL_ONLY"
        gene_id_idx = None
        gene_symbol_idx = 0
        cell_start_idx = 1

    cell_names = cols[cell_start_idx:]
    if len(cell_names) < 10:
        raise ValueError(f"Header parse seems wrong. First 10 cols: {cols[:10]}")

    print(f"[{path_txt.name}] detected format: {fmt} | #cells in header: {len(cell_names)}")

    X_blocks = []
    var_symbol = []
    var_ensembl = []  # may be None for SYMBOL_ONLY formats

    reader = pd.read_csv(path_txt, sep="\t", chunksize=chunk_genes, low_memory=False)

    for i, chunk in enumerate(reader, start=1):
        # gene symbol
        symbol = chunk.iloc[:, gene_symbol_idx].astype(str).values
        var_symbol.extend(symbol.tolist())

        # optional ensembl
        if gene_id_idx is not None:
            ensembl = chunk.iloc[:, gene_id_idx].astype(str).values
            var_ensembl.extend(ensembl.tolist())

        # numeric values: genes x cells
        vals = chunk.iloc[:, cell_start_idx:].to_numpy(dtype=np.float32, copy=False)

        # convert to sparse and transpose -> cells x genes_chunk
        X_blocks.append(sparse.csr_matrix(vals).T)

        if i % 20 == 0:
            print(f"  loaded ~{i*chunk_genes} genes...")

    X = sparse.hstack(X_blocks, format="csr")

    adata = ad.AnnData(X=X)
    adata.obs_names = pd.Index(cell_names, name="cell")
    adata.var_names = pd.Index(var_symbol, name="gene_symbol")
    adata.var_names_make_unique()

    if gene_id_idx is not None:
        adata.var["ensembl_id"] = var_ensembl

    adata.obs = parse_cellname_meta_general(cell_names)
    adata.uns


In [ ]:
OMIX_MSTRT = Path(r"../data/raw/UMI.txt")
OMIX_10X   = Path(r"../data/raw/UMI.txt")

print("Loading Yu mSTRTSeq...")
adata_yu_mstrt = load_omix_umi_txt_sparse_auto(OMIX_MSTRT, chunk_genes=200)
adata_yu_mstrt.obs["dataset"] = "Yu_mSTRTSeq"
print("Yu mSTRT shape:", adata_yu_mstrt.shape)

print("\nLoading Yu 10x...")
adata_yu_10x = load_omix_umi_txt_sparse_auto(OMIX_10X, chunk_genes=200)
adata_yu_10x.obs["dataset"] = "Yu_10x"
print("Yu 10x shape:", adata_yu_10x.shape)


In [ ]:
from pathlib import Path

OMIX_MSTRT = Path(r"../data/raw/UMI.txt")
OMIX_10X   = Path(r"../data/raw/UMI.txt")

def safe_load(label, path, chunk_genes=100):
    print(f"\n=== Loading {label} ===")
    print("Path exists:", path.exists(), "| size (GB):", round(path.stat().st_size/1e9, 2))
    try:
        adata = load_omix_umi_txt_sparse_auto(path, chunk_genes=chunk_genes)
        print("Loaded:", type(adata), "| shape:", adata.shape)
        return adata
    except Exception as e:
        print("FAILED with error:")
        raise  # re-raise so you see full traceback

adata_yu_mstrt = safe_load("Yu mSTRTSeq", OMIX_MSTRT, chunk_genes=100)
adata_yu_10x   = safe_load("Yu 10x", OMIX_10X, chunk_genes=50)  # huge file -> smaller chunk


In [ ]:
import numpy as np
import pandas as pd
import anndata as ad
from pathlib import Path
from scipy import sparse

def parse_cellname_meta_general(cellnames):
    week = []
    sample = []
    tag = []
    for c in cellnames:
        parts = c.split("_", 2)
        if len(parts) >= 2:
            week.append(parts[0])
            sample.append(parts[1])
            tag.append(parts[2] if len(parts) == 3 else None)
        else:
            week.append(None); sample.append(None); tag.append(None)
    return pd.DataFrame({"week": week, "sample": sample, "tag": tag}, index=cellnames)

def load_omix_umi_txt_sparse_auto(path_txt: str | Path, chunk_genes: int = 200) -> ad.AnnData:
    path_txt = Path(path_txt)

    with open(path_txt, "r", encoding="utf-8", errors="ignore") as f:
        header_line = f.readline().rstrip("\n")
    cols = [c.strip().lstrip("\ufeff") for c in header_line.split("\t")]

    if len(cols) >= 3 and cols[0] == "ID" and cols[1] == "Symbol":
        fmt = "ID_SYMBOL"
        gene_id_idx = 0
        gene_symbol_idx = 1
        cell_start_idx = 2
    elif len(cols) >= 2 and cols[0] == "Symbol":
        fmt = "SYMBOL_ONLY"
        gene_id_idx = None
        gene_symbol_idx = 0
        cell_start_idx = 1
    else:
        fmt = "FALLBACK_SYMBOL_ONLY"
        gene_id_idx = None
        gene_symbol_idx = 0
        cell_start_idx = 1

    cell_names = cols[cell_start_idx:]
    if len(cell_names) < 10:
        raise ValueError(f"Header parse seems wrong. First 10 cols: {cols[:10]}")

    print(f"[{path_txt.name}] detected format: {fmt} | #cells in header: {len(cell_names)}")

    X_blocks = []
    var_symbol = []
    var_ensembl = []

    reader = pd.read_csv(path_txt, sep="\t", chunksize=chunk_genes, low_memory=False)

    for i, chunk in enumerate(reader, start=1):
        symbol = chunk.iloc[:, gene_symbol_idx].astype(str).values
        var_symbol.extend(symbol.tolist())

        if gene_id_idx is not None:
            ensembl = chunk.iloc[:, gene_id_idx].astype(str).values
            var_ensembl.extend(ensembl.tolist())

        vals = chunk.iloc[:, cell_start_idx:].to_numpy(dtype=np.float32, copy=False)
        X_blocks.append(sparse.csr_matrix(vals).T)

        if i % 20 == 0:
            print(f"  loaded ~{i*chunk_genes} genes...")

    X = sparse.hstack(X_blocks, format="csr")

    adata = ad.AnnData(X=X)
    adata.obs_names = pd.Index(cell_names, name="cell")
    adata.var_names = pd.Index(var_symbol, name="gene_symbol")
    adata.var_names_make_unique()

    if gene_id_idx is not None:
        adata.var["ensembl_id"] = var_ensembl

    adata.obs = parse_cellname_meta_general(cell_names)
    adata.uns["omix_format"] = fmt

    return adata  # <-- THIS is what was missing in your active version


In [ ]:
from pathlib import Path

OMIX_MSTRT = Path(r"../data/raw/UMI.txt")
OMIX_10X   = Path(r"../data/raw/UMI.txt")

def safe_load(label, path, chunk_genes=100):
    print(f"\n=== Loading {label} ===")
    print("Path exists:", path.exists(), "| size (GB):", round(path.stat().st_size/1e9, 2))
    adata = load_omix_umi_txt_sparse_auto(path, chunk_genes=chunk_genes)
    print("Loaded:", type(adata), "| shape:", adata.shape)
    return adata

adata_yu_mstrt = safe_load("Yu mSTRTSeq", OMIX_MSTRT, chunk_genes=100)
adata_yu_mstrt.obs["dataset"] = "Yu_mSTRTSeq"

adata_yu_10x = safe_load("Yu 10x", OMIX_10X, chunk_genes=50)  # huge -> smaller chunk
adata_yu_10x.obs["dataset"] = "Yu_10x"


In [ ]:
print("Olaniru:", adata_olaniru.shape)
print("Yu mSTRT:", adata_yu_mstrt.shape)
print("Yu 10x:", adata_yu_10x.shape)


In [ ]:
import numpy as np
import scanpy as sc
import anndata as ad

# Make sure dataset labels exist
adata_olaniru.obs["dataset"] = "Olaniru_10x"
adata_yu_mstrt.obs["dataset"] = "Yu_mSTRTSeq"
adata_yu_10x.obs["dataset"] = "Yu_10x"

# Optional: standardize gene symbols a bit
for a in [adata_olaniru, adata_yu_mstrt, adata_yu_10x]:
    a.var_names = a.var_names.astype(str).str.strip()
    a.var_names_make_unique()


In [ ]:
shared_genes = sorted(
    set(adata_olaniru.var_names)
    .intersection(adata_yu_mstrt.var_names)
    .intersection(adata_yu_10x.var_names)
)

print("Shared genes:", len(shared_genes))

adata_olaniru_s = adata_olaniru[:, shared_genes].copy()
adata_yu_mstrt_s = adata_yu_mstrt[:, shared_genes].copy()
adata_yu_10x_s   = adata_yu_10x[:, shared_genes].copy()


In [ ]:
adata_all = ad.concat(
    [adata_olaniru_s, adata_yu_mstrt_s, adata_yu_10x_s],
    join="inner",
    label="batch_source",
    keys=["Olaniru_10x", "Yu_mSTRTSeq", "Yu_10x"]
)

adata_all.var_names_make_unique()
print("adata_all shape:", adata_all.shape)
print(adata_all.obs["batch_source"].value_counts())


In [ ]:
sc.pp.filter_cells(adata_all, min_counts=300)  # keep permissive; fetal immune can be low
sc.pp.filter_genes(adata_all, min_cells=10)

print("After QC:", adata_all.shape)


In [ ]:
pip install scvi-tools


In [ ]:
import scvi

scvi.model.SCVI.setup_anndata(adata_all, batch_key="batch_source")

model = scvi.model.SCVI(adata_all, n_latent=30)
model.train()

adata_all.obsm["X_scVI"] = model.get_latent_representation()


In [ ]:
import scvi
import torch

# (Re-)setup
scvi.model.SCVI.setup_anndata(adata_all, batch_key="batch_source")

model = scvi.model.SCVI(
    adata_all,
    n_latent=30,
)

# Faster training on CPU
model.train(
    max_epochs=30,          # was ~111 by default
    batch_size=512,         # try 512; if memory error, drop to 256
    train_size=0.9,
    early_stopping=True,
    early_stopping_patience=5,
    plan_kwargs={"lr": 1e-3},
    accelerator="cpu",
    num_workers=8,          # try 8; if your laptop struggles, use 4
)


In [ ]:
"adata_all" in globals()


In [ ]:
import anndata as ad

# Make sure dataset labels exist (safe to re-run)
adata_olaniru.obs["dataset"] = "Olaniru_10x"
adata_yu_mstrt.obs["dataset"] = "Yu_mSTRTSeq"
adata_yu_10x.obs["dataset"] = "Yu_10x"

# Find shared genes again (cheap operation)
shared_genes = sorted(
    set(adata_olaniru.var_names)
    .intersection(adata_yu_mstrt.var_names)
    .intersection(adata_yu_10x.var_names)
)

print("Shared genes:", len(shared_genes))

adata_all = ad.concat(
    [
        adata_olaniru[:, shared_genes],
        adata_yu_mstrt[:, shared_genes],
        adata_yu_10x[:, shared_genes],
    ],
    join="inner",
    label="batch_source",
    keys=["Olaniru_10x", "Yu_mSTRTSeq", "Yu_10x"],
)

adata_all.var_names_make_unique()
print("adata_all shape:", adata_all.shape)
print(adata_all.obs["batch_source"].value_counts())


In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from pathlib import Path
from scipy import sparse
DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
OUT_DIR = Path("../outputs")
OMIX_MSTRT = Path(r"../data/raw/UMI.txt")
OMIX_10X   = Path(r"../data/raw/UMI.txt")

PROCESSED_DIR = Path(r"../data/raw/data_processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def load_olaniru_long_matrix_csv(path_gz: str | Path,
                                keep_feature_type: str = "Gene Expression",
                                use_gene_col: str = "feature_name") -> ad.AnnData:
    path_gz = Path(path_gz)
    cols = ["cell_barcode", "feature_id", "feature_name", "feature_type", "count"]

    df = pd.read_csv(
        path_gz,
        compression="gzip",
        sep=",",
        header=None,
        names=cols,
        dtype={"cell_barcode": "string",
               "feature_id": "string",
               "feature_name": "string",
               "feature_type": "string",
               "count": "int32"}
    )

    df["feature_type"] = df["feature_type"].str.strip()
    df["cell_barcode"] = df["cell_barcode"].str.strip()
    df["feature_name"] = df["feature_name"].str.strip()
    df["feature_id"] = df["feature_id"].str.strip()

    df = df[df["feature_type"] == keep_feature_type].copy()

    cells = df["cell_barcode"].astype("category")
    genes = df[use_gene_col].astype("category")

    X = sparse.coo_matrix(
        (df["count"].to_numpy(),
         (cells.cat.codes.to_numpy(), genes.cat.codes.to_numpy())),
        shape=(cells.cat.categories.size, genes.cat.categories.size)
    ).tocsr()

    adata = ad.AnnData(X=X)
    adata.obs_names = cells.cat.categories.astype(str)
    adata.var_names = genes.cat.categories.astype(str)
    adata.var_names_make_unique()
    adata.obs["source_file"] = path_gz.name
    adata.obs["dataset"] = "Olaniru_10x"
    return adata


matrix_files = sorted(GSE_DIR.glob("*_final_matrix.csv.gz"))
adatas = []
keys = []
for f in matrix_files:
    print("Loading Olaniru:", f.name)
    a = load_olaniru_long_matrix_csv(f)
    print("  shape:", a.shape)
    adatas.append(a)
    keys.append(f.stem.replace("_final_matrix", ""))

adata_olaniru = ad.concat(adatas, label="batch", keys=keys, join="outer", fill_value=0)
adata_olaniru.var_names_make_unique()
print("Combined Olaniru shape:", adata_olaniru.shape)

# SAVE checkpoint
olaniru_path = PROCESSED_DIR / "01_olaniru_counts.h5ad"
adata_olaniru.write_h5ad(olaniru_path)
print("Saved:", olaniru_path)


In [ ]:
def parse_cellname_meta_general(cellnames):
    week, sample, tag = [], [], []
    for c in cellnames:
        parts = c.split("_", 2)
        if len(parts) >= 2:
            week.append(parts[0])
            sample.append(parts[1])
            tag.append(parts[2] if len(parts) == 3 else None)
        else:
            week.append(None); sample.append(None); tag.append(None)
    return pd.DataFrame({"week": week, "sample": sample, "tag": tag}, index=cellnames)

def load_omix_umi_txt_sparse_auto(path_txt: str | Path, chunk_genes: int = 100) -> ad.AnnData:
    path_txt = Path(path_txt)

    with open(path_txt, "r", encoding="utf-8", errors="ignore") as f:
        header_line = f.readline().rstrip("\n")
    cols = [c.strip().lstrip("\ufeff") for c in header_line.split("\t")]

    if len(cols) >= 3 and cols[0] == "ID" and cols[1] == "Symbol":
        fmt = "ID_SYMBOL"
        gene_id_idx = 0
        gene_symbol_idx = 1
        cell_start_idx = 2
    elif len(cols) >= 2 and cols[0] == "Symbol":
        fmt = "SYMBOL_ONLY"
        gene_id_idx = None
        gene_symbol_idx = 0
        cell_start_idx = 1
    else:
        fmt = "FALLBACK_SYMBOL_ONLY"
        gene_id_idx = None
        gene_symbol_idx = 0
        cell_start_idx = 1

    cell_names = cols[cell_start_idx:]
    print(f"[{path_txt.name}] format={fmt} | header cells={len(cell_names)}")

    X_blocks = []
    var_symbol = []
    var_ensembl = []

    reader = pd.read_csv(path_txt, sep="\t", chunksize=chunk_genes, low_memory=False)

    for i, chunk in enumerate(reader, start=1):
        symbol = chunk.iloc[:, gene_symbol_idx].astype(str).values
        var_symbol.extend(symbol.tolist())

        if gene_id_idx is not None:
            ensembl = chunk.iloc[:, gene_id_idx].astype(str).values
            var_ensembl.extend(ensembl.tolist())

        vals = chunk.iloc[:, cell_start_idx:].to_numpy(dtype=np.float32, copy=False)
        X_blocks.append(sparse.csr_matrix(vals).T)

        if i % 20 == 0:
            print(f"  loaded ~{i*chunk_genes} genes...")

    X = sparse.hstack(X_blocks, format="csr")

    adata = ad.AnnData(X=X)
    adata.obs_names = pd.Index(cell_names, name="cell")
    adata.var_names = pd.Index(var_symbol, name="gene_symbol")
    adata.var_names_make_unique()

    if gene_id_idx is not None:
        adata.var["ensembl_id"] = var_ensembl

    adata.obs = parse_cellname_meta_general(cell_names)
    adata.uns["omix_format"] = fmt
    return adata


print("Loading Yu mSTRTSeq...")
adata_yu_mstrt = load_omix_umi_txt_sparse_auto(OMIX_MSTRT, chunk_genes=100)
adata_yu_mstrt.obs["dataset"] = "Yu_mSTRTSeq"
print("Yu mSTRT shape:", adata_yu_mstrt.shape)

print("\nLoading Yu 10x...")
adata_yu_10x = load_omix_umi_txt_sparse_auto(OMIX_10X, chunk_genes=50)  # huge file
adata_yu_10x.obs["dataset"] = "Yu_10x"
print("Yu 10x shape:", adata_yu_10x.shape)

# SAVE checkpoints
mstrt_path = PROCESSED_DIR / "02_yu_mstRT_counts.h5ad"
x10_path   = PROCESSED_DIR / "03_yu_10x_counts.h5ad"
adata_yu_mstrt.write_h5ad(mstrt_path)
adata_yu_10x.write_h5ad(x10_path)
print("Saved:", mstrt_path)
print("Saved:", x10_path)


In [ ]:
import anndata as ad
import scanpy as sc
import scvi

# shared genes
shared_genes = sorted(
    set(adata_olaniru.var_names)
    .intersection(adata_yu_mstrt.var_names)
    .intersection(adata_yu_10x.var_names)
)
print("Shared genes:", len(shared_genes))

# concat (inner join since we're using shared_genes anyway)
adata_all = ad.concat(
    [
        adata_olaniru[:, shared_genes],
        adata_yu_mstrt[:, shared_genes],
        adata_yu_10x[:, shared_genes],
    ],
    join="inner",
    label="batch_source",
    keys=["Olaniru_10x", "Yu_mSTRTSeq", "Yu_10x"],
    fill_value=0
)

adata_all.var_names_make_unique()

# IMPORTANT: fix duplicate cell IDs
adata_all.obs_names_make_unique(join="__")

print("adata_all:", adata_all.shape)
print(adata_all.obs["batch_source"].value_counts())

# light QC
sc.pp.filter_cells(adata_all, min_counts=300)
sc.pp.filter_genes(adata_all, min_cells=10)
print("After QC:", adata_all.shape)

# scVI
scvi.model.SCVI.setup_anndata(adata_all, batch_key="batch_source")
model = scvi.model.SCVI(adata_all, n_latent=30)

model.train(
    max_epochs=30,
    batch_size=512,
    early_stopping=True,
    early_stopping_patience=5,
    accelerator="cpu",
)

adata_all.obsm["X_scVI"] = model.get_latent_representation()

# UMAP + clusters
sc.pp.neighbors(adata_all, use_rep="X_scVI", n_neighbors=20)
sc.tl.umap(adata_all)
sc.tl.leiden(adata_all, resolution=1.0)

sc.pl.umap(adata_all, color=["batch_source", "leiden"], frameon=False)



In [ ]:
adata_all.obsm["X_scVI"] = model.get_latent_representation()


In [ ]:
adata_all.write_h5ad(
    r"../data/raw/05_integrated_scvi_latent_umap.h5ad"
)


In [ ]:
import scanpy as sc

adata_all = sc.read_h5ad(
    r"../data/raw/05_integrated_scvi_latent_umap.h5ad"
)


In [ ]:
markers = {
    "Beta": ["INS", "IAPP", "MAFA", "PCSK1"],
    "Alpha": ["GCG", "TTR", "ARX"],
    "Delta": ["SST", "HHEX"],
    "PP": ["PPY"],
    "Endocrine_prog": ["NEUROG3", "NKX2-2", "NEUROD1", "PAX4"],
    "Ductal": ["KRT19", "KRT8", "KRT18", "MUC1", "SOX9"],
    "Acinar": ["PRSS1", "CPA1", "CTRB2", "REG1A"],
    "Endothelial": ["KDR", "ESAM", "PECAM1", "VWF"],
    "Mesenchymal": ["COL1A1", "COL1A2", "DCN", "LUM", "PDGFRA"],
    "Immune": ["PTPRC", "LYZ", "TYROBP"],
    "Macrophage": ["C1QA", "C1QB", "C1QC", "APOE", "MARCO"],
    "Tcell": ["TRAC", "CD3D", "CD3E"],
    "Bcell": ["MS4A1", "CD79A"],
    "NK": ["NKG7", "GNLY"],
}

sc.pl.dotplot(
    adata_all,
    var_names=markers,
    groupby="leiden",
    standard_scale="var",
    dendrogram=True
)


In [ ]:
sc.pl.dotplot(
    adata_all,
    var_names=markers,
    groupby="leiden",
    standard_scale="var",
    dendrogram=True,
    save="_figure1_dendrogram.pdf"
)


In [ ]:
immune = adata_all[adata_all.obs["celltype_major"] == "Immune"].copy()
immune.obs_names_make_unique()

In [ ]:
import scanpy as sc

pan_immune = [g for g in ["PTPRC", "RAC2"] if g in adata_all.var_names]
print("Pan-immune genes used:", pan_immune)

sc.tl.score_genes(adata_all, pan_immune, score_name="pan_immune_score")
sc.pl.violin(adata_all, "pan_immune_score", groupby="leiden", stripplot=False)


In [ ]:
myeloid_mac = [g for g in ["LYZ", "TYROBP", "LST1", "APOE", "C1QA", "C1QB", "C1QC", "MARCO"] if g in adata_all.var_names]
print("Myeloid/mac genes used:", myeloid_mac)

sc.tl.score_genes(adata_all, myeloid_mac, score_name="myeloid_mac_score")
sc.pl.violin(adata_all, "myeloid_mac_score", groupby="leiden", stripplot=False)


In [ ]:
immune = adata_all[adata_all.obs["pan_immune_score"] > 0.2].copy()
immune.obs_names_make_unique()
print("Immune subset:", immune.shape)


In [ ]:
myeloid = immune[immune.obs["myeloid_mac_score"] > 0.2].copy()
myeloid.obs_names_make_unique()
print("Myeloid subset:", myeloid.shape)


In [ ]:
import scanpy as sc

pan_immune = [g for g in ["PTPRC", "RAC2"] if g in adata_all.var_names]
print("Pan-immune genes used:", pan_immune)

sc.tl.score_genes(adata_all, pan_immune, score_name="pan_immune_score")

# quick sanity view
sc.pl.violin(adata_all, "pan_immune_score", groupby="leiden", stripplot=False)


In [ ]:
sc.pp.neighbors(immune, use_rep="X_scVI", n_neighbors=20)
sc.tl.umap(immune)
sc.tl.leiden(immune, resolution=0.8)

sc.pl.umap(immune, color=["leiden"], frameon=False)


In [ ]:
print("adata_all n_obs:", adata_all.n_obs)
print("pan_immune_score in obs?", "pan_immune_score" in adata_all.obs.columns)
print("pan_immune_score summary:")
print(adata_all.obs["pan_immune_score"].describe())

print("How many cells pass cutoff 0.2?",
      (adata_all.obs["pan_immune_score"] > 0.2).sum())

print("immune exists?", "immune" in globals())
if "immune" in globals():
    print("immune n_obs:", immune.n_obs)
    print("X_scVI in immune.obsm?", "X_scVI" in immune.obsm)
    if "X_scVI" in immune.obsm:
        print("X_scVI shape:", immune.obsm["X_scVI"].shape)
    print("X_umap in immune.obsm?", "X_umap" in immune.obsm)
    print("leiden in immune.obs?", "leiden" in immune.obs.columns)


In [ ]:
sc.pl.umap(
    immune,
    frameon=False,
    size=15
)


In [ ]:
sc.pl.umap(
    immune,
    color="leiden",
    frameon=False,
    size=15,
    legend_loc="on data"
)


In [ ]:
sc.pl.umap(
    immune,
    color=["LYZ", "TYROBP"],
    frameon=False,
    size=15
)


In [ ]:
sc.pl.umap(
    immune,
    frameon=False,
    size=20,         # bigger points
    alpha=0.8        # more visible
)


In [ ]:
sc.pl.umap(immune, frameon=False, size=30, alpha=1.0)


In [ ]:
myeloid_markers = {
    "Pan_immune": ["PTPRC", "RAC2"],
    "Myeloid_core": ["LYZ", "TYROBP", "LST1"],
    "Monocyte_like": ["FCN1", "VCAN", "LGALS3", "S100A8", "S100A9"],
    "Macrophage_like": ["C1QA", "C1QB", "C1QC", "APOE", "CTSS", "MARCO"],
    "APC_module": ["HLA-DRA", "CD74", "FCER1A"],
    "Prolif": ["MKI67", "TOP2A"],
    "LPAR5": ["LPAR5"],
}

# keep only genes present
myeloid_markers = {k: [g for g in v if g in immune.var_names] for k, v in myeloid_markers.items()}
print({k: v for k, v in myeloid_markers.items() if len(v)==0})  # shows empty groups, if any

sc.pl.dotplot(
    immune,
    myeloid_markers,
    groupby="leiden",
    dendrogram=True,
    standard_scale="var"
)


In [ ]:
import scanpy as sc

# Make sure leiden is categorical (usually is, but just in case)
immune.obs["leiden"] = immune.obs["leiden"].astype("category")

# Recompute dendrogram for the current immune leiden
sc.tl.dendrogram(immune, groupby="leiden", use_rep="X_scVI")


In [ ]:
sc.pl.dotplot(
    immune,
    myeloid_markers,
    groupby="leiden",
    dendrogram=True,
    standard_scale="var"
)


In [ ]:
import scanpy as sc
import pandas as pd

mac_genes = [g for g in ["C1QA","C1QB","C1QC","APOE","CTSS","MARCO","LYZ","TYROBP","LST1"] if g in immune.var_names]
apc_genes = [g for g in ["HLA-DRA","CD74","FCER1A"] if g in immune.var_names]
cycle_genes = [g for g in ["MKI67","TOP2A"] if g in immune.var_names]

sc.tl.score_genes(immune, mac_genes, score_name="mac_score")
sc.tl.score_genes(immune, apc_genes, score_name="apc_score")
sc.tl.score_genes(immune, cycle_genes, score_name="cycle_score")

scores = immune.obs[["leiden","mac_score","apc_score","cycle_score"]].groupby("leiden").mean()
scores.loc[["23"]].T  # focus on cluster 23


In [ ]:
scores.loc[["10","12","20","23"]].sort_values("mac_score", ascending=False)


In [ ]:
immune_leiden_to_type = {
    # Monocyte-like
    "3": "Monocyte_like",
    "6": "Monocyte_like",
    "14": "Monocyte_like",
    "17": "Monocyte_like",

    # Macrophage-like
    "20": "Macrophage_like",

    # APC-like macrophages / DC-like
    "10": "APC_like_myeloid",
    "12": "APC_like_myeloid",
    "23": "Cycling_APC_like_myeloid",

    # Pure APC
    "5": "APC_like_myeloid",
    "21": "APC_like_myeloid",
}

immune.obs["immune_subtype"] = (
    immune.obs["leiden"]
    .map(immune_leiden_to_type)
    .fillna("Immune_other")
    .astype("category")
)

sc.pl.umap(immune, color="immune_subtype", frameon=False, size=20)


In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd

immune_leiden_to_type = {
    # Monocyte-like
    "3": "Monocyte_like",
    "6": "Monocyte_like",
    "14": "Monocyte_like",
    "17": "Monocyte_like",

    # Macrophage-like (more “classic”)
    "20": "Macrophage_like",

    # APC-like myeloid (mac/APC spectrum)
    "10": "APC_like_myeloid",
    "12": "APC_like_myeloid",
    "5":  "APC_like_myeloid",
    "21": "APC_like_myeloid",

    # Cycling APC-like
    "23": "Cycling_APC_like_myeloid",
}

immune.obs["immune_subtype"] = (
    immune.obs["leiden"].astype(str)
    .map(immune_leiden_to_type)
    .fillna("Immune_other")
    .astype("category")
)

sc.pl.umap(immune, color=["immune_subtype"], frameon=False, size=20)


In [ ]:
if "LPAR5" in immune.var_names:
    sc.pl.violin(immune, "LPAR5", groupby="immune_subtype", stripplot=False, rotation=45)
    sc.pl.boxplot(immune, "LPAR5", groupby="immune_subtype", rotation=45)
else:
    print("LPAR5 not found in immune.var_names")


In [ ]:
# Remove cell-cycle genes and re-check correlation
import numpy as np
import pandas as pd

df = immune.obs.copy()
df["LPAR5"] = immune[:, "LPAR5"].X.toarray().ravel()

# Correlate LPAR5 with cycling score
df["cycle_score"] = immune.obs["cycle_score"]

df[["LPAR5", "cycle_score"]].corr()


In [ ]:
sc.pl.violin(
    immune,
    "LPAR5",
    groupby="immune_subtype",
    stripplot=False,
    order=[
        "Macrophage_like",
        "APC_like_myeloid",
        "Cycling_APC_like_myeloid",
        "Monocyte_like",
        "Immune_other",
    ],
    rotation=45,
)


In [ ]:
immune.write_h5ad(
    r"../data/raw/09_immune_subclustered_annotated.h5ad"
)


In [ ]:
adata_all.write_h5ad(
    r"../data/raw/10_full_pancreas_with_immune_subtypes.h5ad"
)


In [ ]:
summary.to_csv(
    r"../data/raw/LPAR5_by_immune_subtype.csv"
)


In [ ]:
import numpy as np
import pandas as pd

def get_gene_vec(a, gene):
    x = a[:, gene].X
    if hasattr(x, "toarray"):
        x = x.toarray()
    return np.ravel(x)

lpar5 = get_gene_vec(immune, "LPAR5")
df = pd.DataFrame({
    "immune_subtype": immune.obs["immune_subtype"].astype(str).values,
    "LPAR5": lpar5,
    "LPAR5_pos": (lpar5 > 0).astype(int),
})

summary = df.groupby("immune_subtype").agg(
    n=("LPAR5", "size"),
    mean_LPAR5=("LPAR5", "mean"),
    pct_LPAR5_pos=("LPAR5_pos", "mean"),
)
summary["pct_LPAR5_pos"] *= 100

summary.to_csv(
    r"../data/raw/LPAR5_by_immune_subtype.csv"
)


In [ ]:
sc.pl.umap(
    immune,
    color="immune_subtype",
    frameon=False,
    size=20,
    save="_FINAL_immune_umap_subtypes.png"
)

sc.pl.umap(
    immune,
    color="LPAR5",
    frameon=False,
    size=20,
    save="_FINAL_immune_umap_LPAR5.png"
)


In [ ]:
with open(
    r"../data/raw/CHECKPOINT_README.txt",
    "w"
) as f:
    f.write(
        "Checkpoint summary:\n"
        "09_immune_subclustered_annotated.h5ad:\n"
        "- Immune-only cells\n"
        "- Monocyte-like, Macrophage-like, APC-like, Cycling APC-like myeloid\n"
        "- LPAR5 enriched in Cycling APC-like myeloid\n\n"
        "10_full_pancreas_with_immune_subtypes.h5ad:\n"
        "- Full pancreas atlas\n"
        "- Immune subtypes mapped back\n"
        "- Ready for spatial + NicheNet\n"
    )


In [ ]:
import numpy as np
import scanpy as sc

# define LPAR5-high threshold (top quartile within cycling APC-like)
subset = immune.obs["immune_subtype"] == "Cycling_APC_like_myeloid"
lpar5_vals = immune[subset, "LPAR5"].X
if hasattr(lpar5_vals, "toarray"):
    lpar5_vals = lpar5_vals.toarray().ravel()
else:
    lpar5_vals = lpar5_vals.ravel()

thr = np.quantile(lpar5_vals, 0.75)
thr


In [ ]:
immune.obs["LPAR5_high"] = "LPAR5_low"
immune.obs.loc[
    (immune.obs["immune_subtype"] == "Cycling_APC_like_myeloid") &
    (immune[:, "LPAR5"].X.toarray().ravel() >= thr),
    "LPAR5_high"
] = "LPAR5_high"

immune.obs["LPAR5_high"] = immune.obs["LPAR5_high"].astype("category")


In [ ]:
# initialize column
adata_all.obs["LPAR5_high_myeloid"] = "Other_cells"

# transfer labels
adata_all.obs.loc[
    immune.obs_names,
    "LPAR5_high_myeloid"
] = immune.obs["LPAR5_high"].values

adata_all.obs["LPAR5_high_myeloid"] = (
    adata_all.obs["LPAR5_high_myeloid"].astype("category")
)


In [ ]:
sc.pl.umap(
    adata_all,
    color="LPAR5_high_myeloid",
    frameon=False,
    size=8
)


In [ ]:
beta_genes = [g for g in ["INS", "IAPP", "MAFA", "PCSK1"] if g in adata_all.var_names]
sc.tl.score_genes(adata_all, beta_genes, score_name="beta_score")


In [ ]:
sc.pl.umap(
    adata_all,
    color=["beta_score"],
    frameon=False,
    size=8
)


In [ ]:
sc.pl.umap(
    adata_all,
    color=["LPAR5_high_myeloid", "beta_score"],
    frameon=False,
    size=8
)


In [ ]:
from sklearn.neighbors import NearestNeighbors

# coordinates
X = adata_all.obsm["X_umap"]

# index sets
idx_lpar5 = np.where(adata_all.obs["LPAR5_high_myeloid"] == "LPAR5_high")[0]
idx_beta = np.where(adata_all.obs["beta_score"] > 0.5)[0]

# nearest beta distance for each LPAR5-high cell
nbrs = NearestNeighbors(n_neighbors=1).fit(X[idx_beta])
distances, _ = nbrs.kneighbors(X[idx_lpar5])

np.median(distances)


In [ ]:
from sklearn.neighbors import NearestNeighbors
import numpy as np

X = adata_all.obsm["X_umap"]

# LPAR5-high cycling APC-like myeloid cells
idx_lpar5 = np.where(adata_all.obs["LPAR5_high_myeloid"] == "LPAR5_high")[0]

# beta-like cells
idx_beta = np.where(adata_all.obs["beta_score"] > 0.5)[0]

# nearest beta neighbor
nbrs = NearestNeighbors(n_neighbors=1).fit(X[idx_beta])

dist_lpar5, _ = nbrs.kneighbors(X[idx_lpar5])

# CONTROL: random immune cells (excluding LPAR5-high)
idx_immune_all = np.where(adata_all.obs["LPAR5_high_myeloid"] != "Other_cells")[0]

idx_ctrl = np.random.choice(
    idx_immune_all,
    size=len(idx_lpar5),
    replace=False
)

dist_ctrl, _ = nbrs.kneighbors(X[idx_ctrl])

print("Median distance to beta cells:")
print("LPAR5-high cycling APC-like myeloid:", np.median(dist_lpar5))
print("Random immune cells:", np.median(dist_ctrl))


In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.neighbors import NearestNeighbors

# ----------------------------
# EDIT THESE PATHS IF NEEDED
# ----------------------------
base = r"../data/raw/pancreas_gpr92_project"
in_immune = os.path.join(base, "data_processed", "09_immune_subclustered_annotated.h5ad")
in_all    = os.path.join(base, "data_processed", "10_full_pancreas_with_immune_subtypes.h5ad")

outdir = os.path.join(base, "data_processed", "Figure1_exports")
os.makedirs(outdir, exist_ok=True)

# ----------------------------
# LOAD
# ----------------------------
immune = sc.read_h5ad(in_immune)
adata_all = sc.read_h5ad(in_all)

# ----------------------------
# Ensure beta_score exists
# ----------------------------
beta_genes = [g for g in ["INS","IAPP","MAFA","PCSK1"] if g in adata_all.var_names]
if "beta_score" not in adata_all.obs.columns:
    sc.tl.score_genes(adata_all, beta_genes, score_name="beta_score")

# ----------------------------
# Ensure immune_subtype exists in immune
# ----------------------------
if "immune_subtype" not in immune.obs.columns:
    raise KeyError("immune_subtype not found in immune.obs. Recreate your immune_subtype labels first.")

# ----------------------------
# Recompute LPAR5_high within Cycling_APC_like_myeloid (top quartile, reproducible)
# ----------------------------
def get_gene_vec(a, gene):
    x = a[:, gene].X
    if hasattr(x, "toarray"):
        x = x.toarray()
    return np.ravel(x)

if "LPAR5" not in immune.var_names:
    raise KeyError("LPAR5 not found in immune.var_names")

mask_cycle_apc = (immune.obs["immune_subtype"].astype(str) == "Cycling_APC_like_myeloid").values
lpar5_cycle = get_gene_vec(immune[mask_cycle_apc], "LPAR5")

if len(lpar5_cycle) == 0:
    raise ValueError("No cells labeled Cycling_APC_like_myeloid in immune_subtype.")

thr = np.quantile(lpar5_cycle, 0.75)

immune.obs["LPAR5_high"] = "LPAR5_low"
immune.obs.loc[mask_cycle_apc & (get_gene_vec(immune, "LPAR5") >= thr), "LPAR5_high"] = "LPAR5_high"
immune.obs["LPAR5_high"] = immune.obs["LPAR5_high"].astype("category")

# ----------------------------
# Project LPAR5_high back to adata_all
# ----------------------------
adata_all.obs["LPAR5_high_myeloid"] = "Other_cells"
adata_all.obs.loc[immune.obs_names, "LPAR5_high_myeloid"] = immune.obs["LPAR5_high"].astype(str).values
adata_all.obs["LPAR5_high_myeloid"] = adata_all.obs["LPAR5_high_myeloid"].astype("category")

# ----------------------------
# Prepare distances for Fig1D
# ----------------------------
X = adata_all.obsm["X_umap"]
idx_lpar5 = np.where(adata_all.obs["LPAR5_high_myeloid"].astype(str) == "LPAR5_high")[0]
idx_beta  = np.where(adata_all.obs["beta_score"].values > 0.5)[0]

if len(idx_lpar5) == 0:
    raise ValueError("No LPAR5_high cells found in adata_all.obs['LPAR5_high_myeloid'].")
if len(idx_beta) == 0:
    raise ValueError("No beta-like cells found using beta_score > 0.5. Try a lower cutoff.")

nbrs = NearestNeighbors(n_neighbors=1).fit(X[idx_beta])
dist_lpar5, _ = nbrs.kneighbors(X[idx_lpar5])
dist_lpar5 = dist_lpar5.ravel()

idx_immune_all = np.where(adata_all.obs["LPAR5_high_myeloid"].astype(str) != "Other_cells")[0]
rng = np.random.default_rng(0)
idx_ctrl = rng.choice(idx_immune_all, size=len(idx_lpar5), replace=False)
dist_ctrl, _ = nbrs.kneighbors(X[idx_ctrl])
dist_ctrl = dist_ctrl.ravel()

# ----------------------------
# Define dotplot marker sets (filtered to existing genes)
# ----------------------------
myeloid_markers = {
    "Pan_immune": ["PTPRC", "RAC2"],
    "Myeloid_core": ["LYZ", "TYROBP", "LST1"],
    "Monocyte_like": ["FCN1", "VCAN", "LGALS3", "S100A8", "S100A9"],
    "Macrophage_like": ["C1QA", "C1QB", "C1QC", "APOE", "CTSS", "MARCO"],
    "APC_module": ["HLA-DRA", "CD74", "FCER1A"],
    "Prolif": ["MKI67", "TOP2A"],
    "LPAR5": ["LPAR5"],
}
myeloid_markers = {k: [g for g in v if g in immune.var_names] for k, v in myeloid_markers.items()}

# ----------------------------
# Create and save panels
# ----------------------------
pngA = os.path.join(outdir, "Fig1A_full_umap_LPAR5_high_myeloid.png")
pngB = os.path.join(outdir, "Fig1B_immune_umap_subtypes.png")
pngC = os.path.join(outdir, "Fig1C_violin_LPAR5_by_immune_subtype.png")
pngD = os.path.join(outdir, "Fig1D_distance_to_beta_LPAR5high_vs_random.png")
pngE = os.path.join(outdir, "Fig1E_dotplot_myeloid_continuum.png")
pdf_all = os.path.join(outdir, "Figure1_combined.pdf")

# --- Fig1A
fig = sc.pl.umap(adata_all, color="LPAR5_high_myeloid", frameon=False, size=8, show=False, return_fig=True)
fig.savefig(pngA, dpi=300, bbox_inches="tight")
plt.close(fig)

# --- Fig1B
fig = sc.pl.umap(immune, color="immune_subtype", frameon=False, size=20, legend_loc="on data", show=False, return_fig=True)
fig.savefig(pngB, dpi=300, bbox_inches="tight")
plt.close(fig)

# --- Fig1C
fig, ax = plt.subplots(figsize=(6.5, 3.5))
sc.pl.violin(immune, keys="LPAR5", groupby="immune_subtype", stripplot=False, rotation=45, ax=ax, show=False)
fig.tight_layout()
fig.savefig(pngC, dpi=300, bbox_inches="tight")
plt.close(fig)

# --- Fig1D
fig, ax = plt.subplots(figsize=(5.2, 3.5))
ax.boxplot([dist_lpar5, dist_ctrl], labels=["LPAR5-high\nCycling APC-like", "Random\nimmune"], showfliers=False)
ax.set_ylabel("Nearest-neighbor distance to beta-like cells (UMAP space)")
ax.set_title(f"Median: {np.median(dist_lpar5):.3f} vs {np.median(dist_ctrl):.3f}")
fig.tight_layout()
fig.savefig(pngD, dpi=300, bbox_inches="tight")
plt.close(fig)

# --- Fig1E dotplot (robust save)
immune.obs["leiden"] = immune.obs["leiden"].astype("category")
sc.tl.dendrogram(immune, groupby="leiden", use_rep="X_scVI")

dp = sc.pl.dotplot(
    immune,
    myeloid_markers,
    groupby="leiden",
    dendrogram=True,
    standard_scale="var",
    show=False,
    return_fig=True
)

# dp is a DotPlot object; get the matplotlib figure and save it
fig = dp.get_axes()["mainplot_ax"].figure
fig.savefig(pngE, dpi=300, bbox_inches="tight")
plt.close(fig)


# ----------------------------
# Combine into a single multi-page PDF
# ----------------------------
with PdfPages(pdf_all) as pdf:
    for p in [pngA, pngB, pngC, pngD, pngE]:
        img = plt.imread(p)
        fig, ax = plt.subplots(figsize=(8.5, 6.5))
        ax.imshow(img)
        ax.axis("off")
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

print("Saved Figure 1 panels to:", outdir)
print("Combined PDF:", pdf_all)
